# Data Cleaning

In [1]:
# Import statements
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

import numpy as np; np.set_printoptions(precision=2)
import matplotlib.pyplot as plt; #plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams.update({ "figure.figsize": [8, 3], "figure.dpi": 125, "text.usetex": True, "font.family": "Helvetica" })
import pandas as pd; pd.options.display.float_format = "{:,.2f}".format
import warnings; warnings.filterwarnings('ignore')

In [2]:
!pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# metadata
print(bank_marketing.metadata)

# variable information
print(bank_marketing.variables)

{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

In [18]:
# Import Statements

import pandas as pd

# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
from sklearn.model_selection import train_test_split

# There was a sepereate Geeks4Geeks article on this to standardize categorical data too
# I can't find the link however
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# https://scikit-learn.org/stable/modules/compose.html
# https://www.geeksforgeeks.org/pipelines-python-and-scikit-learn/
from sklearn.pipeline import Pipeline

# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
from sklearn.linear_model import LogisticRegression

# https://scikit-learn.org/stable/api/sklearn.metrics.html
from sklearn.metrics import accuracy_score, classification_report, precision_score

# find the right link later for citations
from sklearn.tree import DecisionTreeClassifier

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt


In [37]:
import matplotlib
import os

# The command below should list the LaTeX path if it's installed correctly
!which latex

# Point Matplotlib to your newly installed LaTeX
matplotlib.rcParams['text.usetex'] = True
os.environ['PATH'] += os.pathsep + '/usr/bin/'  # Might vary depending on the installation path of latex

/usr/bin/latex


In [10]:
bank_full = pd.read_csv('bank-full.csv', sep = ';')
print(bank_full.head())

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  


In [11]:
# Count NA/NULL values in each column (code from: https://www.geeksforgeeks.org/how-to-count-the-number-of-nan-values-in-pandas/)
column_nan_count = bank_full.isna().sum()
print("NaN count per column:")
print(column_nan_count)

# Appears to be no NA/NULL values in bank_full

NaN count per column:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [12]:
# Copying the entire structure over from the API hints from the midterm - prepping the data

# Convert categorical features to numerical (since I again was copying code from the midterm)
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] # List of categorical columns
numerical_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] # List of numerical columns

# Create a ColumnTransformer to apply different preprocessing to different columns
# Stolen from stackoverflow https://stackoverflow.com/questions/54160370/how-to-use-sklearn-column-transformer
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
# https://www.geeksforgeeks.org/ml-one-hot-encoding/
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features),
    ])

# Feature selection, kind of threw everything in there due to lack of domain knowledge
# And also because we're going to use L2 regularization anyways
features = ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']
y_bank_full = bank_full[['y']]

In [14]:
# Splitting the data into a training/test set

# X_bank_full was not defined, using bank_full[features] instead to select the desired features for the model
X_train, X_test, y_train, y_test = train_test_split(bank_full[features], y_bank_full, test_size=0.2, random_state=42) # Use X_bank_full instead of X

# Fit and transform the training data
X_train_scaled = preprocessor.fit_transform(X_train)

# Transform the test data
X_test_scaled = preprocessor.transform(X_test)

# Decision Tree Attempt:

In [16]:
# Inital Fitting

clf = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", DecisionTreeClassifier(max_depth=100))]
)

clf.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'balance', 'day',
                                                   'duration', 'campaign',
                                                   'pdays', 'previous']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome'])])),
                ('classifier', DecisionTreeClassifier(max_depth=100))])

In [28]:
# Convert 'no' and 'yes' to numerical values 0 and 1
y_test_numeric = y_test['y'].replace({'no': 0, 'yes': 1})

# Plot the probablities
probabilities = clf.predict_proba(X_test)[:, 1]

# Calculate predictions using the trained model
y_pred = clf.predict(X_test) # This line was added to calculate y_pred

from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# Changed code: Specify pos_label='yes' for precision_score
precision = precision_score(y_test, y_pred, pos_label='yes')
recall = recall_score(y_test, y_pred, pos_label='yes')

print(f'Accuracy: {accuracy:.4f}')
print('Confusion Matrix:')
print(conf_matrix)
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

Accuracy: 0.8761
Confusion Matrix:
[[7386  566]
 [ 554  537]]
Precision: 0.4869
Recall: 0.4922


In [45]:
# Altered Lecture Code in order to not plot everything every time

def print_feature_importances(clf, numeric_cols, cat_cols):
    """
    Print feature importances from a sklearn pipeline with decision tree classifier

    Args:
        clf: Fitted sklearn pipeline with decision tree classifier
        numeric_cols: List of numeric column names
        cat_cols: List of categorical column names
    """
    # Get feature names after preprocessing
    feature_names = (numeric_cols +
                     clf.named_steps['preprocessor']
                     .named_transformers_['cat']
                     .get_feature_names_out(cat_cols).tolist())

    # Get feature importances from the classifier
    importances = clf.named_steps['classifier'].feature_importances_

    # Create DataFrame of features and their importances
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    })

    # Sort by importance
    importance_df = importance_df.sort_values('importance', ascending=False)

    # Print feature importances
    print("\nFeature Importances:")
    print(importance_df)

# Usage
print_feature_importances(clf, numerical_features, categorical_features)


Feature Importances:
                feature  importance
3              duration        0.27
1               balance        0.11
49     poutcome_success        0.09
0                   age        0.09
2                   day        0.09
5                 pdays        0.05
4              campaign        0.03
6              previous        0.02
41            month_jun        0.01
42            month_mar        0.01
35            month_apr        0.01
29          housing_yes        0.01
34      contact_unknown        0.01
23  education_secondary        0.01
43            month_may        0.01
45            month_oct        0.01
16       job_technician        0.01
7            job_admin.        0.01
36            month_aug        0.01
20      marital_married        0.01
44            month_nov        0.01
8       job_blue-collar        0.01
11       job_management        0.01
28           housing_no        0.01
38            month_feb        0.01
21       marital_single        0.01
40    

# Post Pruning:

In [46]:
# Taken from the lecture code, this is the Cost-Complexity Pruning (CCP) Alpha

# First, let's create a function to find the optimal ccp_alpha using the validation set
def optimize_ccp_alpha(pipeline, X_train, X_val, y_train, y_val):
    """
    Find optimal ccp_alpha for decision tree using cost complexity pruning.

    Args:
        pipeline: Sklearn pipeline with preprocessor and classifier
        X_train, X_val: Training and validation features
        y_train, y_val: Training and validation labels

    Returns:
        best_alpha: Optimal complexity parameter
        best_score: Validation accuracy at best alpha
    """
    # Preprocess both training and validation data
    preprocessor = pipeline.named_steps['preprocessor']
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    # Get path for different values of ccp_alpha
    tree = DecisionTreeClassifier()
    path = tree.cost_complexity_pruning_path(X_train_processed, y_train)
    ccp_alphas = path.ccp_alphas

    # Remove the maximum alpha that would give a tree with just the root
    ccp_alphas = ccp_alphas[:-1]

    # Store accuracy scores
    best_alpha = 0
    best_score = 0

    # Try different alpha values
    for ccp_alpha in ccp_alphas:
        # Create and train tree with current alpha
        tree = DecisionTreeClassifier(ccp_alpha=ccp_alpha)
        tree.fit(X_train_processed, y_train)

        # Evaluate on validation set
        score = tree.score(X_val_processed, y_val)

        # Update best if improved
        if score > best_score:
            best_score = score
            best_alpha = ccp_alpha

    return best_alpha, best_score

In [47]:
from sklearn.model_selection import train_test_split

# Assuming bank_full is your DataFrame containing the data
X_bank_full = bank_full[features]  # Use the features defined earlier
y_bank_full = bank_full['y']  # Assuming 'y' is the target column

# Split the data into training and test sets
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_bank_full,
    y_bank_full,
    test_size=0.2,
    random_state=42
)

# Further split the training set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42
)


In [49]:
# Create initial pipeline
initial_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier())
    ]
)

In [50]:
best_alpha, best_val_score = optimize_ccp_alpha(
    initial_pipeline,
    X_train,
    X_val,
    y_train,
    y_val
)

print(f"Best ccp_alpha: {best_alpha:.6f}")
print(f"Validation accuracy: {best_val_score:.3f}")

Best ccp_alpha: 0.000161
Validation accuracy: 0.907


In [51]:
# Create final pipeline with optimal ccp_alpha
final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(ccp_alpha=best_alpha))
    ]
)

# Train final model on combined train+validation data
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
final_pipeline.fit(X_train_full, y_train_full)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'balance', 'day',
                                                   'duration', 'campaign',
                                                   'pdays', 'previous']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome'])])),
                ('classifier',
                 DecisionTreeClassifier(ccp_alpha=0.00016125489312337945))])

In [53]:
from sklearn.metrics import recall_score

# Evaluate on test set
test_score = final_pipeline.score(X_test, y_test)
print(f"Test accuracy: {test_score:.3f}")

# Make predictions on the test set
y_pred = final_pipeline.predict(X_test)

# Check the distribution of the true labels
print("True labels distribution:")
print(y_test.value_counts())

# Check the unique values in predictions
print("Predicted labels unique values:")
print(np.unique(y_pred))

# Calculate recall
try:
    # Specify the positive label as 'yes'
    recall = recall_score(y_test, y_pred, pos_label='yes')
    print(f"Recall: {recall:.3f}")
except ValueError as e:
    print(f"Error calculating recall: {e}")


Test accuracy: 0.900
True labels distribution:
y
no     7952
yes    1091
Name: count, dtype: int64
Predicted labels unique values:
['no' 'yes']
Recall: 0.490
